# 10 — SHAP / XAI por territorio funcional (corrida limpia)

Re-calcula SHAP sobre los modelos **v2** (pipeline de encoding corregido, commit
`d8fbe44`) y completa el comentario contenido 2 (Figura 6):

1. **Importancia relativa**: share de |SHAP| por feature y clúster.
2. **Intervalos bootstrap 95 %** del share (200 remuestreos de filas).
3. **SHAP con signo**: media del SHAP value por feature (dirección del efecto)
   + beeswarm por clúster.
4. **Matrices crudas** de SHAP en `shap_results/raw/*.npz` (sin recalcular).

**Fuentes de datos requeridas:** dataset `lucasiturriago/pruebassaber11` +
notebook `lucasiturriago/12-diagnostico-mbe-saberxai` (aporta `models/` v2).
**Acelerador:** GPU T4 x2 (`NvidiaTeslaT4`), Internet activado.

In [ ]:
!git clone https://github.com/liturriago/EquidadAlgoritmica-SaberXAI.git
%cd EquidadAlgoritmica-SaberXAI
!pip install -e . -q

In [ ]:
# Resolver dataset y modelos (los modelos vienen del OUTPUT del notebook 12, como kernel source)
from glob import glob
from pathlib import Path
import subprocess

print(subprocess.run(['ls', '/kaggle/input'], capture_output=True, text=True).stdout)

cands = [
    Path('/kaggle/input/pruebassaber11'),
    Path('/kaggle/input/datasets/lucasiturriago/pruebassaber11'),
]
data_dir = next((p for p in cands if (p / 'clase_0.parquet').exists()), None)
if data_dir is None:
    import kagglehub
    data_dir = Path(kagglehub.dataset_download('lucasiturriago/pruebassaber11'))
print('data_dir =', data_dir)

hits = sorted(glob('/kaggle/input/**/xgb_cluster_6.json', recursive=True))
assert hits, "No se encontró xgb_cluster_6.json bajo /kaggle/input. Adjunta el notebook 12 como fuente de datos (kernel source)."
models_dir = Path(hits[0]).parent
print('models_dir =', models_dir)

# Helper: ejecuta un comando y aborta el notebook si falla (los !python no abortan)
import shlex
import subprocess

def run(cmd):
    print(">>", cmd)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"Falló (exit {r.returncode}): {cmd}"


In [ ]:
# SHAP por clúster: XGBoost y MLP, matrices crudas + bootstrap + signo
run(f"python scripts/shap_analysis.py --config configs/shap_analysis.yaml --data-dir {shlex.quote(str(data_dir))} --models-dir {shlex.quote(str(models_dir))} --model-type both")

In [ ]:
# Top-10 features por clúster (share de |SHAP| con IC 95 %) — XGBoost
import pandas as pd
from IPython.display import display

df = pd.read_csv('shap_results/shap_importance_XGBoost.csv')
for c in sorted(df['cluster_id'].unique()):
    top = df[df['cluster_id'] == c].nlargest(10, 'share')
    print(f"\n=== Clúster {c} ===")
    display(top[['feature', 'share', 'share_ci_low', 'share_ci_high', 'mean_shap_signed']].round(4))